In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# Load dataset
df = pd.read_csv('/content/telco_churn.csv')

# Drop customerID
df = df.drop('customerID', axis=1)

# Convert TotalCharges to numeric
df['TotalCharges'] = df['TotalCharges'].replace(" ", "0")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])

print("Total customers after cleaning:", df.shape[0])

df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
# Fix: Drop rows where 'Churn' might have become NaN after mapping
df.dropna(subset=['Churn'], inplace=True)

# Convert categorical to numeric
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object' and col != 'Churn':
        df[col] = le.fit_transform(df[col])

# Split data
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fix: Scale data *after* splitting to prevent data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fix: Define and Train model (added solver and max_iter for stability)
model = LogisticRegression(max_iter=2000, solver='liblinear')
model.fit(X_train_scaled, y_train)

# Predict on scaled test data
y_pred = model.predict(X_test_scaled)

# Results
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Total customers after cleaning: 5043
Accuracy: 0.7701711491442543
Confusion Matrix:
 [[260  28]
 [ 66  55]]


The model prioritizes overall accuracy, but future improvements can focus on increasing recall for churned customers

In [26]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

dt = DecisionTreeClassifier()
rf = RandomForestClassifier()

dt.fit(X_train, y_train)
rf.fit(X_train, y_train)

print(f"DT Train Accuracy: {accuracy_score(y_train, dt.predict(X_train))*100:.2f}%")
print(f"DT Test Accuracy: {accuracy_score(y_test, dt.predict(X_test))*100:.2f}%")
print(f"RF Accuracy: {accuracy_score(y_test, rf.predict(X_test))*100:.2f}%")
print("RF Accuracy:", accuracy_score(y_test, rf.predict(X_test)))

DT Train Accuracy: 100.00%
DT Test Accuracy: 67.97%
RF Accuracy: 76.28%
RF Accuracy: 0.7628361858190709


Logistic Regression:

In [6]:
from sklearn.metrics import roc_auc_score
y_prob = model.predict_proba(X_test_scaled)[:,1]
auc = roc_auc_score(y_test, y_prob)

print(f"AUC: {auc:.2f}")

AUC: 0.83


Decision Tree:

In [7]:
dt_prob_full = dt.predict_proba(X_test)
print(f"DT AUC: {roc_auc_score(y_test, dt_prob_full[:, 1]):.2f}")

DT AUC: 0.61


Random Forest:

In [8]:
rf_prob_full = rf.predict_proba(X_test)
print(f"RF AUC: {roc_auc_score(y_test, rf_prob_full[:, 1]):.2f}")

RF AUC: 0.81
